In [1]:
import os
import time
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pathlib as pl
import shutil
import sys
import pandas as pd

import flopy
from modflowapi import ModflowApi
from modflowapi.extensions import ApiSimulation

from bmi.wrapper import BMIWrapper

import pyswmm
from pyswmm import Simulation, Nodes
from pyswmm import Output

c:\Users\bbayrakt\AppData\Local\miniforge3\envs\liss\lib\site-packages\bmi\__init__.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
c:\Users\bbayrakt\AppData\Local\miniforge3\envs\liss\lib\site-packages\pkg_resources\__init__.py:2832: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('pydap')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
c:\Users\bbayrakt\AppData\Local\miniforge3\envs\liss\lib\site-packages\pkg_resources\__init__.py:2832: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('pydap.responses')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/e

In [2]:
sys.path.append("../common")
from liss_settings import \
    libmf6, \
    get_dflow_grid_name, get_dflow_dtuser, \
    get_modflow_coupling_tag, get_modflow_grid_name, \
    silent, verbosity, \
    print_path, print_value

# Load Names

## dflow

In [3]:
control_path = pl.Path("../dflow-fm/highres/tides_atm_surge_2018/FlowFM.mdu") # change this if using a different D-Flow FM control file
grid_name = get_dflow_grid_name(control_path)
print(grid_name)
dflowfm_dtuser = get_dflow_dtuser(control_path)
print(f'{dflowfm_dtuser} seconds')

LIS_HighestRES_nvd88_net
60.0 seconds


## modflow

In [4]:
mf_grid_name = get_modflow_grid_name()

#  Unit Conversions

In [5]:
d2sec = 24. * 60. * 60.
hrs2sec = 60. * 60. 
m2ft = 3.28081
cfd2cms = 1.0 / ((m2ft**3) * 86400.)

# MODFLOW coupling frequency

Change the `mf_couple_freq_hours` value. Only tested for multiple of the D-Flow FM DtUser variable. Will not work for `mf_couple_freq_hours` values greater than 24.

In [6]:
mf_couple_freq_hours = 0.25 # Change this value to change the coupling frequency
mf_couple_freq = mf_couple_freq_hours * hrs2sec
dflow_per_mf = int(mf_couple_freq / dflowfm_dtuser)
print(f"MODFLOW coupling frequency {mf_couple_freq_hours} hours\nMODFLOW coupled to D-FLOW FM every {dflow_per_mf} output time step ({dflowfm_dtuser} sec.)") 

mf_tag = get_modflow_coupling_tag(mf_couple_freq_hours)
print(f"MODFLOW coupling tag: {mf_tag}")

mf_couple_nstp = int(86400.0 / (dflow_per_mf * dflowfm_dtuser))
print(f"MODFLOW time steps per day: {mf_couple_nstp }")

MODFLOW coupling frequency 0.25 hours
MODFLOW coupled to D-FLOW FM every 15 output time step (60.0 sec.)
MODFLOW coupling tag: 15.00M
MODFLOW time steps per day: 96


# Set a few variables for controlling coupling

In [7]:
HDRY = -1e30
DEPTH_MIN = 0.1

#### Print the path of the modflow6 shared library

In [8]:
str(libmf6), libmf6.is_file()

('C:\\Users\\bbayrakt\\AppData\\Local\\miniforge3\\envs\\liss\\Scripts\\libmf6.dll',
 False)

# D-FLOW to MODFLOW weights


## GHB weights

In [9]:
fpath =  f"../mapping/PJ/dflow{grid_name}_to_{mf_grid_name}_ghb.npz"
npzfile = np.load(fpath)
print(fpath)
dflow2mfghb = npzfile["dflow2mfghb"]
print(f'dflow2ghb shape: {dflow2mfghb.shape}')
ghbmask = npzfile["ghbmask"]
print(f'ghb mask shape :{ghbmask.shape}')
ghb2qext = npzfile["ghb2qext"]
print(f'ghb2qext shape: {ghb2qext.shape}')

../mapping/PJ/dflowLIS_HighestRES_nvd88_net_to_PJmf6_ghb.npz
dflow2ghb shape: (354, 54243)
ghb mask shape :(354,)
ghb2qext shape: (54243, 354)


## CHD weights

In [10]:
fpath = f"../mapping/PJ/dflow{grid_name}_to_{mf_grid_name}_chd.npz"

print(fpath)
npzfile = np.load(fpath)
dflow2mfchd = npzfile["dflow2mfchd"]
print(f'dflow2chd shape: {dflow2mfchd.shape}')
chdmask = npzfile["chdmask"]
print(f'chd mask shape :{chdmask.shape}')
chd2qext = npzfile["chd2qext"]
print(f'chd2qext shape: {chd2qext.shape}')

../mapping/PJ/dflowLIS_HighestRES_nvd88_net_to_PJmf6_chd.npz
dflow2chd shape: (1461, 54243)
chd mask shape :(1461,)
chd2qext shape: (54243, 1461)


# Create Modflow Model

In [11]:
# set path to gwf model (uncoupled)
mf_base_path = pl.Path("../modflow/pj_2018_adjust_FINAL_NoRecharge/base/").resolve()
# create directory to run coupled gwf model
mf_run_path = pl.Path(f"../modflow/pj_2018_adjust_FINAL_NoRecharge/MF+DFlowNEWGRID/run_{mf_tag}/").resolve()


In [12]:
sim = flopy.mf6.MFSimulation.load(sim_ws=mf_base_path, verbosity_level=verbosity())
gwf = sim.get_model()

In [13]:
sim.set_sim_path(mf_run_path)
# create outputs in mf_run_path
(mf_run_path / 'outputs').mkdir(exist_ok=True)

## Write the new model files

In [14]:
sim.write_simulation(silent=silent())

# Define base GHB variables

In [15]:
ghb_data0 = gwf.ghb.stress_period_data.get_dataframe()[0]
assert ghb_data0.shape[0] == ghbmask.shape[0]

# Define base CHD variables

2 CHD packages
1. Surface CHDs (chds in bay, and perimeter chd's in bay ,ALL in the top layer) which only has one external file
2. Perimeter CHDs (perimeter chds not in the bay, at all layers), 366 external files for every day. 

In [16]:
chd_all = gwf.get_package("chd_coast")
chd_data0 = chd_all.stress_period_data.get_dataframe()[0]
# print_value(chd_data0)
assert chd_data0.shape[0] == chdmask.shape[0]
chd_all.stress_period_data.get_dataframe()[0]

,cellid_layer,cellid_row,cellid_column,head,salinity,boundname
0,0,3,19,0.330,33.9,bay
1,0,3,20,0.330,34.6,bay
2,0,3,21,0.330,34.9,bay
3,0,3,22,0.330,34.7,bay
4,0,3,23,0.330,34.7,bay
...,...,...,...,...,...,...
1456,0,22,68,0.324,34.2,surface perimeter coastal
1457,0,24,70,0.324,34.4,surface perimeter coastal
1458,0,26,72,0.324,34.4,surface perimeter coastal
1459,0,28,74,0.324,34.4,surface perimeter coastal


# Setup and initialize D-FLOW FM
You will need to set `dflow_dirpath` to the correct directory on your machine.

## Paths

In [17]:
dflow_dirpath = pl.Path(r"..\dflow-fm\dflowfm_dll").resolve()
dflow_base = pl.Path(r"../dflow-fm/highres/tides_atm_surge_2018/").resolve()
dflow_working = pl.Path(r"../dflow-fm/highres/tides_atm_surge_2018/run_mfNoRecharge").resolve()
dflow_config = dflow_working / "FlowFM.mdu"
print(dflow_config)

D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\highres\tides_atm_surge_2018\run_mfNoRecharge\FlowFM.mdu


In [18]:
if dflow_working.is_dir():
    shutil.rmtree(dflow_working)
shutil.copytree(dflow_base, dflow_working,   ignore=shutil.ignore_patterns("run"))
(dflow_working / "output").mkdir(parents=True, exist_ok=True)

In [19]:
# Add dflowfm dll folder to PATH so that it can be found by the BMIWrapper
os.environ["PATH"] = (
    str(dflow_dirpath) + os.pathsep + os.environ["PATH"]
)

In [20]:
(pl.Path(dflow_dirpath) / "dflowfm.dll").is_file()

True

## Initialize D-Flow FM API

In [21]:
dflowfm = BMIWrapper(
    engine="dflowfm",
    configfile=str(dflow_config),
)

In [22]:
import os
print("cwd:", os.getcwd())

cwd: d:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\notebooks-PJ


In [23]:
dflowfm.initialize()# this line of code is what changes the cwd to  '\dflow-fm\coarse\tides\run'

In [24]:
import os
print("cwd:", os.getcwd())

cwd: D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\highres\tides_atm_surge_2018\run_mfNoRecharge


## Get data using DFLOW API

In [25]:
ndxi = int(dflowfm.get_var("ndxi"))
ndx = int(dflowfm.get_var("ndx")) # number of nodes
x = dflowfm.get_var("xz")
y = dflowfm.get_var("yz")
z = dflowfm.get_var("bl")
xy = [(xx, yy) for (xx, yy) in zip(x, y)]

ndx, ndxi, x.shape, y.shape # bnb note: are these values ok? answer: they shouldnt matter because they run with the gp model 

(54332, 54243, (54332,), (54332,))

In [26]:
def read_mdu(path):
    config = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):  # skip comments
                continue
            if "=" in line:
                key, val = [s.strip() for s in line.split("=", 1)]
                config[key.lower()] = val
    return config

mdu_data = read_mdu(dflow_config)
dflow_start_date = mdu_data["refdate"].split()[0]
dflow_start_date = datetime.datetime.strptime(dflow_start_date, "%Y%m%d")
dflow_end_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_end_time())
print(f'DFlow start date: {dflow_start_date} \nDFlow end date: {dflow_end_date}')

DFlow start date: 2018-09-17 00:00:00 
DFlow end date: 2018-10-17 00:00:00


In [27]:
qext = np.zeros(ndx)
qext.shape, qext

((54332,), array([0., 0., 0., ..., 0., 0., 0.]))

In [28]:
qext_cum = np.zeros(ndx)
qext_cum.shape

(54332,)

In [29]:
vextcum = dflowfm.get_var("vextcum")
vextcum.shape, vextcum

((54332,), array([0., 0., 0., ..., 0., 0., 0.]))

# Initialize MODFLOW using MODFLOW API

## Change Modflow TDIS

In [30]:
tdis = sim.get_package("TDIS")
mf_perioddata= tdis.perioddata.array
print(mf_perioddata)
mf6_start_date = datetime.datetime.strptime(tdis.start_date_time.get_data(), "%Y-%m-%d")
# calculate date for each stress period
mf_SPdates = [mf6_start_date]
for perlen, _, _ in mf_perioddata:
    mf_SPdates.append(mf_SPdates[-1] + pd.Timedelta(days = perlen))
# remove the last date because it is extra
mf_SPdates = mf_SPdates[:-1]

# Compute cumulative total times at the end of each stress period
mf_totaltimes= tdis.perioddata.array['perlen'].cumsum()

mf_tdis_df = pd.DataFrame({'Date':mf_SPdates,
                           'SP_data': list(mf_perioddata),
                           'SP': [i for i in range(len(mf_perioddata))],
                           'totim': mf_totaltimes})

[(365., 3, 1.3) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. ) (  1., 1, 1. )
 (  1., 1, 1. ) (  1., 1

In [31]:
# latest date among models
latest_date = max(dflow_start_date, mf6_start_date)
print("Latest date:", latest_date)

# find closest available date in mf_tdis_df
closest_date_idx = (mf_tdis_df['Date'] - latest_date).abs().idxmin() 
print(closest_date_idx)
closest_date = mf_tdis_df.loc[closest_date_idx, 'Date']

# get corresponding stress period
latest_date_SP = mf_tdis_df.loc[closest_date_idx, 'SP']

print("Closest available date:", closest_date)
print("Associated SP:", latest_date_SP)

print(f'Modflow Stress Period associated with the lastest models start date: {latest_date_SP}')
assert latest_date_SP + len(mf_perioddata[latest_date_SP:]) == len(mf_tdis_df)
# Update nstp where SP index >= latest_date_SP
mf_perioddata["nstp"][latest_date_SP:] = mf_couple_nstp 
mf_perioddata

mf_tdis_df['SP_data_coupled'] = list(mf_perioddata)


# CHANGE TDIS IN MF MODEL TO REFLECT TIME STEP CHANGES
tdis.mf_perioddata = mf_perioddata
print(mf_perioddata)


Latest date: 2018-09-17 00:00:00
260
Closest available date: 2018-09-17 00:00:00
Associated SP: 260
Modflow Stress Period associated with the lastest models start date: 260
[(365.,  3, 1.3) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. )
 (  1.,  1, 1. ) (  1.,  1, 1. ) (  1.,  1, 1. 

## OC

In [32]:
# dates of interst
dates_oc = ['2018-01-01','2018-02-01','2018-03-01','2018-04-01','2018-05-01','2018-06-01', '2018-07-01',
            '2018-08-01','2018-09-01','2018-09-24','2018-09-25','2018-9-26','2018-09-27', '2018-09-28','2018-09-29',
            '2018-09-30', '2018-10-01','2018-11-01', '2018-12-01'  ]
# save SPs of interest for output control file
sp_start = []
sp_end = []
for date in dates_oc:
    # identify stress period
    sp = mf_tdis_df.loc[mf_tdis_df['Date']== date, 'SP'].iloc[0]
    sp_start.append(sp)
    sp_1 = sp+1
    sp_end.append(sp_1)
    print(date, sp,sp_1)

# create dictionary for output control file
oc_dict = {}
for date , i in zip(dates_oc, sp_start):
    if date == '2018-09-25':
        oc_dict[i] = [('head','all')]
    else:
        oc_dict[i] = [('head','last')]

for i in sp_end:
    if i in sp_start:
        continue
    else:
        oc_dict[i] = []
# sort dictionary
oc_dict = dict(sorted(oc_dict.items()))
oc_gwf = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord='outputs/gwf.cbc',
    head_filerecord='outputs/gwf.hds',
    saverecord=oc_dict
)

2018-01-01 1 2
2018-02-01 32 33
2018-03-01 60 61
2018-04-01 91 92
2018-05-01 121 122
2018-06-01 152 153
2018-07-01 182 183
2018-08-01 213 214
2018-09-01 244 245
2018-09-24 267 268
2018-09-25 268 269
2018-9-26 269 270
2018-09-27 270 271
2018-09-28 271 272
2018-09-29 272 273
2018-09-30 273 274
2018-10-01 274 275
2018-11-01 305 306
2018-12-01 335 336


In [33]:
sim.write_simulation(silent=silent())

## MF API

In [34]:
# set the mdll_path to be the absolute path where the mf6 dll is located
#mdll_path = pl.Path(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\mf6dll\libmf6.dll")
mdll_path = str(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\mf6dll\libmf6.dll")
print(mdll_path)

mf_run_path = str(mf_run_path)
print(mf_run_path)

D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\mf6dll\libmf6.dll
D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\pj_2018_adjust_FINAL_NoRecharge\MF+DFlowNEWGRID\run_15.00M


In [35]:
mf6 = ModflowApi(mdll_path, working_directory=mf_run_path)

In [36]:
mf6.initialize()

# Define Variables and Functions for coupling

## Define MODFLOW variable tags and set pointer to MODFLOW variables

In [37]:
ghb_bhead_tag = mf6.get_var_address("BHEAD", "GWF", "GHB")
ghb_cond_tag = mf6.get_var_address("COND", "GWF", "GHB")
ghb_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "GHB")

In [38]:
ghb_bhead_ptr = mf6.get_value_ptr(ghb_bhead_tag)
ghb_cond_ptr = mf6.get_value_ptr(ghb_cond_tag)
ghb_flow = np.zeros(ghb_bhead_ptr.shape)

In [39]:
chd_head_tag = mf6.get_var_address("HEAD", "GWF", "CHD_coast")
chd_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "CHD_coast")

In [40]:
chd_head_ptr = mf6.get_value_ptr(chd_head_tag)
chd_flow = np.zeros(chd_head_ptr.shape)

## Create dictionaries for saving modified GHB data

In [41]:
ghb_elev_dict = {}
ghb_cond_dict = {}
chd_elev_dict = {}
qext_dict = {}


## Function to update MODFLOW GHB & CHD data

In [42]:
def update_mf(key, s, d):
    #print('running update_mf')
    mask = d == 0.0
    s[mask] = 0.0
    mult = np.full(d.shape, 1.0)
    mult[mask] = 0.0

    ghb_head = ghb_data0["bhead"].to_numpy()
    # print(f'ghb_data0 : {ghb_data0["bhead"].shape}') 
    # print(f'ghb_head :{ghb_head.shape}')

    ghb_head[ghbmask] = dflow2mfghb.dot(s)[ghbmask] * m2ft
    # print(f'ghbmask : {ghbmask.shape} , dflow2mfghb :{dflow2mfghb.shape}')

    ghb_cond = ghb_data0["cond"].to_numpy()
    # print(f'ghb_cond : {ghb_data0["cond"].shape}') 

    ghb_cond[ghbmask] = ghb_cond[ghbmask] * dflow2mfghb.dot(mult)[ghbmask]
    
    ghb_bhead_ptr[:] = ghb_head[:] # does not work
    #ghb_bhead_ptr[:] = ghb_bhead_ptr[:] * 1.5 # _ptr variable is a pointer for the APR. 

    ghb_cond_ptr[:] = ghb_cond[:] #does not work
    #ghb_cond_ptr[:] = ghb_cond_ptr[:]
    
    chd_head = chd_data0["head"].to_numpy()

    chd_head[chdmask] = dflow2mfchd.dot(s)[chdmask] * m2ft
    
    chd_head_ptr[:] = chd_head[:] # does not work
    #chd_head_ptr[:] = chd_head_ptr[:] * 1.5 #this works
   
    # update results dictionary
    ghb_elev_dict[key] = ghb_head.copy()
    ghb_cond_dict[key] = ghb_cond.copy()
    chd_elev_dict[key] = chd_head.copy()

## Function to update D-Flow FM Qext data

In [43]:
def update_dflow(key, d):
    ghb_flow = -mf6.get_value(ghb_flow_tag) * cfd2cms
    #print(f'ghb_flow: {ghb_flow.shape, mf6.get_value(ghb_flow_tag).shape}')
    
    dflow_qext_ghb = ghb2qext.dot(ghb_flow)
    #print(f'dflow_qext_ghb{dflow_qext_ghb.shape, ghb2qext.shape, ghb_flow.shape}')

    dflow_qext_ghb[d == 0.0] = 0.0
    
    chd_flow = -mf6.get_value(chd_flow_tag) * cfd2cms
    #print(f'chd_flow{chd_flow.shape, mf6.get_value(chd_flow_tag).shape}')

    dflow_qext_chd = chd2qext.dot(chd_flow)

    dflow_qext_chd[d == 0.0] = 0.0

    dflow_qext = dflow_qext_ghb + dflow_qext_chd
    
    qext_cum[:ndxi] += dflow_qext[:ndxi]
    qext[:ndxi] = dflow_qext[:ndxi]
    dflowfm.set_var("qext", qext)

    # update results dictionaries
    qext_dict[key] = qext[:ndxi].copy()

# Run each time step

In [44]:
print(
    f"DFLOWFM current_time: {dflowfm.get_current_time():15,.1f} sec. ({dflowfm.get_current_time()/86400.:15,.1f} days)\n"
     + f"DFLOWFM end_time:     {dflowfm.get_end_time():15,.1f} sec. ({dflowfm.get_end_time()/86400.:15,.1f} days)"
)

DFLOWFM current_time:             0.0 sec. (            0.0 days)
DFLOWFM end_time:         2,592,000.0 sec. (           30.0 days)


In [45]:
import datetime
mf6_start_date = datetime.datetime.strptime(tdis.start_date_time.get_data(), "%Y-%m-%d")
mf6_end_date = mf6_start_date + datetime.timedelta(days=mf6.get_end_time())
print(f'MF start date: {mf6_start_date} \nMF end date: {mf6_end_date}')
print(f'Dflow start date: {dflow_start_date}')
print(f'Dflow end date: {dflow_end_date}')

MF start date: 2017-01-01 00:00:00 
MF end date: 2019-01-01 00:00:00
Dflow start date: 2018-09-17 00:00:00
Dflow end date: 2018-10-17 00:00:00


In [46]:
#======================================================================
# THIS CELL COUPLES TO THE TWO MODELS AFTER DFLOW HAS A SPINUP PERIOD PRIOD TO COUPLING
#=======================================================================
idx = 0
jdx = 0
t0 = time.perf_counter()

# NOTE: we'll assume that the modflow model will run the longest

mf_current_date = mf6_start_date
couple_start_date = datetime.datetime.fromisoformat('2018-09-20 00:00:00')

# if dflow_start_date<couple_start_date:
#     print("running dflow spinup")

if mf6_start_date < couple_start_date:
    print("Initializing Modflow Sim")
    #==============================================================
    # STEP 1 : Run MF until one of the other two models is "ready"
    #=============================================================
    while mf_current_date < couple_start_date:
        mf6.update()
        mf_current_date = mf6_start_date + datetime.timedelta(days=mf6.get_current_time())
        print(mf_current_date)
    assert mf_current_date == couple_start_date
    print(f'End of Step 1a---\nMF:{mf_current_date}')
    print('---------------------------------')

if dflow_start_date<couple_start_date:
    print("Initializing Dflow Spinup")
    dflow_current_date = dflow_start_date
    while dflow_current_date<couple_start_date:
        dflowfm.update()
        dflow_current_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_current_time())
        print(dflow_current_date)
    assert dflow_current_date == couple_start_date
    print(f'End of Step 1b---\nMF:{dflow_current_date}')
    print('---------------------------------')


if dflow_current_date == mf_current_date:
    print('Now coupling MF + DFLOW together')
    while dflow_current_date <= dflow_end_date:
        idx += 1
        #frac_comp = 1 - (dflow_end_date - current_date).days / (dflow_end_date - start_date).days
        #(f"(Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
        dflowfm.update()
        dflow_current_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_current_time())
        #print(dflow_current_date)

        if idx == int(dflow_per_mf):
            print(f"(*Coupling* - Current date: {dflow_current_date})")
           # print(f"(*Coupling* - Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
            s = dflowfm.get_var("s1")[:ndxi] # water level
            d = dflowfm.get_var("hs")[:ndxi]
            
            mf6.prepare_time_step(mf6.get_time_step())
            mf_current_date = mf6_start_date + datetime.timedelta(days=mf6.get_current_time())
            print(mf_current_date,dflow_current_date)
            update_mf(str(jdx), s, d)
            mf6.do_time_step()
            mf6.finalize_time_step()
            update_dflow(str(jdx), d)
            
            # # advance SWMM
            # update_swmm(str(jdx))
            # dt_sec = mf6.get_time_step() * d2sec
            # swmm_sim.step_advance(int(dt_sec))
            # try:
            #     swmm_sim.__next__()  
            # except StopIteration:
            #     break        

            # update counters
            idx = 0
            jdx += 1
        
        if dflow_current_date >= dflow_end_date:
            break

vextcum = dflowfm.get_var("vextcum")

t1 = time.perf_counter()
print(f"\nrun time: {(t1 - t0) / 60.} min")

Initializing Modflow Sim
2017-04-02 11:29:19.398496
2017-07-30 09:37:26.616541
2018-01-01 00:00:00
2018-01-02 00:00:00
2018-01-03 00:00:00
2018-01-04 00:00:00
2018-01-05 00:00:00
2018-01-06 00:00:00
2018-01-07 00:00:00
2018-01-08 00:00:00
2018-01-09 00:00:00
2018-01-10 00:00:00
2018-01-11 00:00:00
2018-01-12 00:00:00
2018-01-13 00:00:00
2018-01-14 00:00:00
2018-01-15 00:00:00
2018-01-16 00:00:00
2018-01-17 00:00:00
2018-01-18 00:00:00
2018-01-19 00:00:00
2018-01-20 00:00:00
2018-01-21 00:00:00
2018-01-22 00:00:00
2018-01-23 00:00:00
2018-01-24 00:00:00
2018-01-25 00:00:00
2018-01-26 00:00:00
2018-01-27 00:00:00
2018-01-28 00:00:00
2018-01-29 00:00:00
2018-01-30 00:00:00
2018-01-31 00:00:00
2018-02-01 00:00:00
2018-02-02 00:00:00
2018-02-03 00:00:00
2018-02-04 00:00:00
2018-02-05 00:00:00
2018-02-06 00:00:00
2018-02-07 00:00:00
2018-02-08 00:00:00
2018-02-09 00:00:00
2018-02-10 00:00:00
2018-02-11 00:00:00
2018-02-12 00:00:00
2018-02-13 00:00:00
2018-02-14 00:00:00
2018-02-15 00:00:00
2

In [47]:
#=================================================================================================================================
# THIS CELL COUPLES TO THE TWO MODELS ONCE THE DFLOW START DATE = MODFLOW CURRENT DATE. THERE IS NO DFLOW SPINUP PRIOR TO COUPLING
#==================================================================================================================================
# idx = 0
# jdx = 0
# t0 = time.perf_counter()

# # NOTE: we'll assume that the modflow model will run the longest
# start_date = mf6_start_date
# current_date = mf6_start_date


# if mf6_start_date < dflow_start_date:
#     print("Initializing Modflow Sim")
#     #==============================================================
#     # STEP 1 : Run MF until one of the other two models is "ready"
#     #=============================================================
#     while current_date < dflow_start_date:
#         mf6.update()
#         current_date = start_date + datetime.timedelta(days=mf6.get_current_time())
#         print(current_date)
#     assert current_date == dflow_start_date
#     print(f'End of Step 1---\nMF:{current_date} \nDflow:{dflow_start_date}')
#     print('---------------------------------')
    
#     print('Now coupling MF + DFLOW together')
#     while current_date <= dflow_end_date:
#         idx += 1
#         #frac_comp = 1 - (dflow_end_date - current_date).days / (dflow_end_date - start_date).days
#         #(f"(Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
#         dflowfm.update()
#         current_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_current_time())
#         print(current_date)
#         if idx == int(dflow_per_mf):
#             print(f"(*Coupling* - Current date: {current_date})")
#            # print(f"(*Coupling* - Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
#             s = dflowfm.get_var("s1")[:ndxi] # water level
#             d = dflowfm.get_var("hs")[:ndxi]
            
#             mf6.prepare_time_step(mf6.get_time_step())
#             update_mf(str(jdx), s, d)
#             mf6.do_time_step()
#             mf6.finalize_time_step()
#             update_dflow(str(jdx), d)
            
#             # # advance SWMM
#             # update_swmm(str(jdx))
#             # dt_sec = mf6.get_time_step() * d2sec
#             # swmm_sim.step_advance(int(dt_sec))
#             # try:
#             #     swmm_sim.__next__()  
#             # except StopIteration:
#             #     break        

#             # update counters
#             idx = 0
#             jdx += 1
        
#         if current_date >= dflow_end_date:
#             break

# vextcum = dflowfm.get_var("vextcum")

# t1 = time.perf_counter()
# print(f"\nrun time: {(t1 - t0) / 60.} min")

In [48]:
# print(f'MF start date: {mf6_start_date}')
# print(f'Dflow start date: {dflow_start_date}')
# print('-------------------------------------')

# idx = 0
# jdx = 0
# t0 = time.perf_counter()

# # NOTE: we'll assume that the modflow model will run the longest
# start_date = mf6_start_date
# current_date = mf6_start_date


# if mf6_start_date < dflow_start_date:
#     print("Initializing Modflow Sim")
#     #==============================================================
#     # STEP 1 : Run MF until one of the other two models is "ready"
#     #=============================================================
#     while current_date < dflow_start_date:
#         mf6.update()
#         current_date = start_date + datetime.timedelta(days=mf6.get_current_time())
#         print(current_date)
#     assert current_date == dflow_start_date
#     print(f'End of Step 1---\nMF:{current_date} \nDflow:{dflow_start_date}')
#     print('---------------------------------')
    
#     print('Now coupling MF + DFLOW together')
#     while current_date <= dflow_end_date:
#         idx += 1
#         frac_comp = 1 - (dflow_end_date - current_date).days / (dflow_end_date - start_date).days
#         print(f"(Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
#         dflowfm.update()
#         current_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_current_time())

#         if idx == int(dflow_per_mf):
#             print(f"(*Coupling* - Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
#             s = dflowfm.get_var("s1")[:ndxi] # water level
#             d = dflowfm.get_var("hs")[:ndxi]
            
#             mf6.prepare_time_step(mf6.get_time_step())
#             update_mf(str(jdx), s, d)
#             mf6.do_time_step()
#             mf6.finalize_time_step()
#             update_dflow(str(jdx), d)
            
#             # # advance SWMM
#             # update_swmm(str(jdx))
#             # dt_sec = mf6.get_time_step() * d2sec
#             # swmm_sim.step_advance(int(dt_sec))
#             # try:
#             #     swmm_sim.__next__()  
#             # except StopIteration:
#             #     break        

#             # update counters
#             idx = 0
#             jdx += 1
        
#         if current_date >= dflow_end_date:
#             mf6.update()
#             current_date = start_date + datetime.timedelta(days=mf6.get_current_time())
#             print(current_date)  

# vextcum = dflowfm.get_var("vextcum")

# t1 = time.perf_counter()
# print(f"\nrun time: {(t1 - t0) / 60.} min")

#### Finalize models

In [49]:
mf6.finalize()

In [50]:
# swmm_sim.terminate_simulation()
# swmm_sim.report()
# swmm_sim.close()

In [51]:
dflowfm.finalize()

#### Save ghb elevation and conductance data to compressed files

In [52]:
np.savez_compressed(f"{mf_run_path}/ghb_elev.npz", **ghb_elev_dict)
np.savez_compressed(f"{mf_run_path}/ghb_cond.npz", **ghb_cond_dict)

#### Save chd elevation to compressed file

In [53]:
np.savez_compressed(f"{mf_run_path}/chd_elev.npz", **chd_elev_dict)

#### Save qext data to compressed file

In [54]:
np.savez_compressed(f"{mf_run_path}/qext.npz", **qext_dict)

#### Save SWMM flux data to compressed file

In [55]:
# np.savez_compressed(f"{mf_run_path}/swmm_q.npz", **swmm_q_dict)